# Prepare Kaggle Dataset from Google Drive

This notebook:
1. Mounts Google Drive
2. Copies aligned_utterances.jsonl, prosody_phaseD.json, and all audio/*.mp3
3. Cleans audio paths in the JSONL
4. Zips everything for Kaggle upload

Output: `chuckle_kaggle_dataset.zip` on Google Drive

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE = '/content/drive/MyDrive/chuckle_checkpoints'
AUDIO_DIR = f'{BASE}/audio'
PROSODY_PATH = f'{BASE}/prosody_phaseD.json'
ALIGNED_PATH = f'{BASE}/aligned_utterances.jsonl'
OUTPUT_DIR = '/content/kaggle_dataset'
ZIP_PATH = '/content/drive/MyDrive/chuckle_kaggle_dataset'

import os, json, shutil
from pathlib import Path
# Clean previous run to avoid accumulating stale files
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(f'{OUTPUT_DIR}/audio', exist_ok=True)
print('Ready')


In [ ]:
# Step 1: Clean and copy aligned_utterances.jsonl
print('Processing aligned_utterances.jsonl...')
count = 0
with open(ALIGNED_PATH) as fin, open(f'{OUTPUT_DIR}/aligned_utterances.jsonl', 'w') as fout:
    for line in fin:
        d = json.loads(line)
        d['audio_file'] = f"{d['video_id']}.mp3"
        fout.write(json.dumps(d) + '\n')
        count += 1
print(f'  Wrote {count} entries ✅')


In [ ]:
# Step 2: Copy prosody_phaseD.json
print('Copying prosody_phaseD.json...')
shutil.copy2(PROSODY_PATH, f'{OUTPUT_DIR}/prosody_phaseD.json')
size = os.path.getsize(f'{OUTPUT_DIR}/prosody_phaseD.json') / 1024**2
print(f'  Copied ({size:.1f} MB) ✅')


In [ ]:
# Step 3: Copy all audio files
print('Copying audio files...')
audio_files = sorted(Path(AUDIO_DIR).glob('*.mp3'))
total = 0
for i, src in enumerate(audio_files):
    shutil.copy2(src, f'{OUTPUT_DIR}/audio/{src.name}')
    total += src.stat().st_size
    if (i+1) % 10 == 0:
        print(f'  {i+1}/{len(audio_files)} ({total/1024**3:.1f} GB)')
print(f'  Total: {len(audio_files)} files, {total/1024**3:.1f} GB ✅')


In [ ]:
# Step 4: Create Kaggle metadata
metadata = {
    'title': 'chuckle-net-phase-a-prosody',
    'id': 'YOUR-USERNAME/chuckle-net-phase-a-prosody'  # ← Replace YOUR-USERNAME with your Kaggle username,
    'licenses': [{'name': 'CC0-1.0'}]
}
with open(f'{OUTPUT_DIR}/dataset-metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('  Metadata created ✅')


In [ ]:
# Step 5: Zip it up
print('Creating zip (this may take a few minutes)...')
shutil.make_archive(ZIP_PATH, 'zip', OUTPUT_DIR)
zip_size = os.path.getsize(f'{ZIP_PATH}.zip') / 1024**3
print(f'  Created: {ZIP_PATH}.zip ({zip_size:.1f} GB) ✅')

print('\n=== Next Steps ===')
print(f'1. Download {ZIP_PATH}.zip from Google Drive')
print('2. Go to kaggle.com/datasets/create')
print('3. Upload the zip')
print('4. Create new notebook, add dataset via + Add Data')
print('5. Run ChuckleNet_Kaggle.ipynb')
